In [1]:
# Creating a mask for country specific BMR

In [2]:
import xarray as xr
import numpy as np

In [3]:
# === Path config ===
BMR_DIR = "/glade/work/awells/air_quality/BMR/"
MASKS_DIR = "/glade/work/awells/air_quality/BMR/masks/country/"

In [4]:
# BMR from GBD (https://vizhub.healthdata.org/gbd-results/) is provided as a
# RATE per 100,000.
# To calculate the mortality in "total number of people" we must divide the rate
# by 100,000 in order to convert the rate per 100,000 into the rate per 1.

In [5]:
# Divide rate by 100,000 since rate is num per 100k
bmr = xr.open_dataarray(f"{BMR_DIR}GBD_BMR_Country_COPD_1990-2009.nc") / 100000

# Use country masks to create a BMR mask for each country
masks = xr.open_dataarray(f"{MASKS_DIR}GBD_Country_Masks_0.10_newlabels.nc")

In [6]:
# Create DataArray filled with NaNs
global_bmr_array = xr.DataArray(
    np.full((len(masks.lat), len(masks.lon)), np.nan),
    coords={"lat": masks.lat.values, "lon": masks.lon.values},
    dims=["lat", "lon"]
)

In [7]:
# Loop over countries, select the BMR for each country and apply to empty array
# using the country mask
for i in range(len(masks.country)):
    mask = masks.isel(country=i)
    country = masks.isel(country=i)["country"]
    bmr_country = bmr.sel(country=country)  # adds all three quantiles
    global_bmr_array = global_bmr_array.where(mask == 0, bmr_country)

In [8]:
# Save BMR mask
description = ("Mean Baseline Mortality Rate per country for COPD from"
               "1990-2009 as a global mask")
cite = ("Global Burden of Disease Collaborative Network. Global Burden of"
        "Disease Study 2021 (GBD 2021) Results. Seattle, United States: "
        "Institute for Health Metrics and Evaluation (IHME), 2022. Available "
        "from https://vizhub.healthdata.org/gbd-results/.")

global_bmr_array.attrs["description"] = description
global_bmr_array.attrs["citation"] = cite
global_bmr_array.to_netcdf(f"{BMR_DIR}GBD_BMR_Country_Mask_COPD_1990-2009.nc")